# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [1]:
# Imports (DO NOT EDIT)
import sys, os
sys.path.append(".")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface
from ase.visualize import view

import site_analysis as sa

In [2]:
# USER INPUT
name = 'pd332.xyz'
n_layers = 10
atoms = surface('Pd',(3,3,2),n_layers)
atoms.center(vacuum=10, axis=2)
adsorbate_height = 2.
site_bond_cutoff = 3.

In [3]:
#visualize slab
write(name, atoms)
view(atoms, viewer='x3d')

In [4]:

# Load slab
slab = read(name)
surface = CustomSurface(slab, n_layers=n_layers)
nslab = len(slab)


In [18]:

slabrat = slab.copy()
slabrat.rattle(stdev=0.3)

# Generate symmetry-unique site geometries
cas = SlabAdsorptionSites(slab, "fcc332", composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slab, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slabrat, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!

single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

print(f'There are {len(single_sites_lists)} unique sites out of {len(cas.get_sites())}.')
#print(cas.get_sites())

traj = Trajectory("unique_sites.traj", "w")
for g in single_geoms:
    traj.write(g)
traj.close()


There are 33 unique sites out of 72.


In [8]:
[cas.get_sites()[i]['morphology'] for i in range(len(cas.get_sites()))]

['subsurf',
 'subsurf',
 'subsurf',
 'subsurf',
 'terrace',
 'sc-tc',
 'sc-tc',
 'sc-tc',
 'sc-tc',
 'sc-tc',
 'terrace',
 'terrace',
 'terrace',
 'terrace',
 'step',
 'step',
 'step',
 'sc-tc',
 'sc-tc',
 'step',
 'sc-tc',
 'terrace']

In [166]:

# Extract 3-fold site graphs
admols, threefold_geom_indices = sa.classify_threefold_sites(
    single_geoms, single_sites_lists
)


In [167]:

# Graph isomorphism clustering
iso_mat, clusters = sa.cluster_isomorphic_graphs(admols)


In [168]:

# Update 3-fold site labels
type_map = sa.update_threefold_site_labels(
    single_sites_lists,
    clusters,
    threefold_geom_indices
)


In [169]:

# Write 3-fold-only trajectory
traj3 = Trajectory("threefold_sites.traj", "w")
for i in threefold_geom_indices:
    traj3.write(single_geoms[i])
traj3.close()


In [170]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


Pairwise strict isomorphism:
0 vs 1 = False
0 vs 2 = False
1 vs 2 = False


In [171]:

# Distinct 3-fold site types (PRINT)
print("Number of distinct 3-fold site types:", len(clusters))
for members in clusters.values():
    print("3-fold site type:", members)


Number of distinct 3-fold site types: 3
3-fold site type: [0]
3-fold site type: [1]
3-fold site type: [2]


In [172]:

# Updated 3-fold site labels (PRINT)
print("Updated 3-fold site labels per geometry:")
for geom_idx, label in type_map.items():
    print(f"Geometry {geom_idx} -> {label}")


Updated 3-fold site labels per geometry:
Geometry 3 -> 3fold1
Geometry 4 -> 3fold2
Geometry 7 -> 3fold3


In [173]:
# Site yaml file generated
print("All sites for the custom surfaces are saved in site.yaml")
sa.write_sites_yaml(single_sites_lists, clusters)


All sites for the custom surfaces are saved in site.yaml
